# Kill test: does the image model beat the cell table? (`future_plan.md` §0.0)

**No training.** The three existing Jackson → Danenberg checkpoints are used: the joint ResNet-UNet without HyperConv, one per preprocessing arm (residual, gamma, immuvis), 11 shared markers.

**GPU pass.**
- For every (arm, marker), predict the marker on each Danenberg image.
- With the published cell masks (`danenberg-cells-prep`, relabelled to `ObjectNumber`), average per cell, in the model's target space:
  - the true target;
  - the prediction;
  - 5 cyclically shifted predictions (null: realistic texture, wrong place);
  - one cyclically shifted true map (control).
- Per-image pixel sums give the pooled pixel r used in E2.
- A CPU pass averages the raw image channels per cell, so every cell-level feature comes from the same uncorrected images the model sees.

**Cell-table baselines.** Trained on the Jackson cell table, applied to the Danenberg image-derived cell means; cross-cohort, like the image model.
- **B1:** the cell's own 10 shared inputs + H3, DNA1, DNA2 + log area.
- **B2:** B1 plus the mean of the same inputs over non-touching cells within 30 µm.
- **B1r:** B1 with rank (quantile) harmonisation instead of median / p5–p95.

**Rungs** (Danenberg cells in tumour images; patient-bootstrap CIs, 1000×):
- **pooled r:** pixel Pearson, as in E2.
- **R4:** per-cell Pearson after removing each image's mean.
- **R5_input / R5_panel:** within-image Pearson between the residuals of truth and of prediction, each after removing what the non-target markers imply.
  - The "state" part is a cross-fitted gradient-boosting regression: fitted on one half of the patients, applied to the other half.
  - The non-target markers are the model's own inputs (R5_input) or the whole Danenberg panel (R5_panel).
  - This replaces the plan's kNN mean with its smooth equivalent.

**Positive controls** (true data; they certify R4/R5):
- cyclically shifted true map: R4 ≈ 0;
- state mean (the truth's regression on the inputs): R4 > 0, R5 ≈ 0;
- truth shuffled within image among cells of the same state decile: R4 > 0, R5 ≈ 0;
- identity: 1.

**Downstream readouts.** Frozen split: fitted on train, scored on val; test is left untouched.
- ER / HER2 status AUROC from the patient mean over tumour cells; train + val patients.
- Phenotype-classifier substitution: macro-F1 change on val cells when marker M is replaced by its imputed version (rank-normalised features).
- M+ fraction per (val patient × broad class): r between measured and imputed, with thresholds from train cells.

**Decisions (fixed before the run).**
- **Image beats table:** R4(arm) − R4(B1) has a CI > 0 for ≥ 2 markers in at least one arm. B2 is also reported.
- **Beyond-panel signal:** R5_input(arm) − R5_input(shifted arm) has a CI > 0 for ≥ 2 markers.
- **D1:** for ≥ 2 markers, some pair of arms ranks in opposite directions by pooled r and by R4, both CIs excluding 0.
- **D2:** adding R4 + R5_input to pooled r raises the leave-one-marker-out R² of a cell-level readout (F1 change or M+ agreement) by ≥ 0.1 (33 units).
- **Outcome** (§0.0 table):
  - dissociation (D1 or D2) and beyond-panel → **GO**;
  - dissociation only → **PIVOT-A**;
  - beyond-panel only → **PIVOT-B**;
  - neither → **KILL**.
- **Caveat:** the published masks used membrane channels; this is acceptable for this test only.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Model code (copied unchanged from `AbundanceVsStructure.ipynb`) and the fitted preprocessing

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast
from torchvision import models
from scipy.signal import butter, sosfiltfilt
from sklearn.linear_model import SGDRegressor
from sklearn.decomposition import IncrementalPCA, PCA

# ─── Shared building blocks (used by all conditions) ─────────────────────────

class ResNet50Encoder(nn.Module):
    def __init__(self, in_channels=3, pretrained_conv1=False):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        if pretrained_conv1 and in_channels == 3:
            self.conv1 = backbone.conv1
        else:
            self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        for param in backbone.parameters():
            param.requires_grad = False
        # re-enable conv1 after the freeze loop if using pretrained weights
        if pretrained_conv1 and in_channels == 3:
            for param in self.conv1.parameters():
                param.requires_grad = True
        for param in self.layer2.parameters():
            param.requires_grad = True
        for param in self.layer3.parameters():
            param.requires_grad = True
        for param in self.layer4.parameters():
            param.requires_grad = True

    def forward(self, x):
        f0 = self.relu(self.bn1(self.conv1(x)))
        f0_pool = self.maxpool(f0)
        f1 = self.layer1(f0_pool)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return f0, f1, f2, f3, f4


class UNetDecoder(nn.Module):
    def __init__(self, out_channels=1, p=0.2):
        super().__init__()
        self.conv4_1 = nn.Sequential(
            nn.Conv2d(2048+1024, 1024, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(1024), nn.ReLU(inplace=True), nn.Dropout2d(p))
        self.conv4_2 = nn.Sequential(
            nn.Conv2d(1024, 1024, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(1024), nn.ReLU(inplace=True))
        self.conv3_1 = nn.Sequential(
            nn.Conv2d(1024+512, 512, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True), nn.Dropout2d(p))
        self.conv3_2 = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True))
        self.conv2_1 = nn.Sequential(
            nn.Conv2d(512+256, 256, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Dropout2d(p))
        self.conv2_2 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.conv1_1 = nn.Sequential(
            nn.Conv2d(256+256+64, 64, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.Dropout2d(p))
        self.conv1_2 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.up0 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1, padding_mode='reflect', bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.final = weight_norm(nn.Conv2d(32, out_channels, 1))

    def forward(self, skips):
        f0, f1, f2, f3, f4 = skips
        x = F.interpolate(f4, scale_factor=2, mode='bilinear', align_corners=False)
        if x.shape[-2:] != f3.shape[-2:]:
            f3 = F.interpolate(f3, size=x.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv4_2(self.conv4_1(torch.cat([x, f3], 1)))
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if x.shape[-2:] != f2.shape[-2:]:
            f2 = F.interpolate(f2, size=x.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv3_2(self.conv3_1(torch.cat([x, f2], 1)))
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if x.shape[-2:] != f1.shape[-2:]:
            f1 = F.interpolate(f1, size=x.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv2_2(self.conv2_1(torch.cat([x, f1], 1)))
        if f0.shape[-2:] != f1.shape[-2:]:
            f0 = F.interpolate(f0, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv1_2(self.conv1_1(torch.cat([x, f1, f0], 1)))
        x = F.interpolate(self.up0(self.final_up(x)), scale_factor=2, mode='bilinear', align_corners=False)
        return torch.sigmoid(self.final(x))


class ResNetUNet(nn.Module):
    def __init__(self, in_channels=3, pretrained_conv1=False):
        super().__init__()
        self.encoder = ResNet50Encoder(in_channels=in_channels, pretrained_conv1=pretrained_conv1)
        self.decoder = UNetDecoder()

    def forward(self, x, marker_ids=None):   # marker_ids ignored — marker-agnostic baseline
        return self.decoder(self.encoder(x))


# ─── Hyperconv encoder (ImmuVis-style, arxiv:2602.04585) ─────────────────────
#
# MarkerStem replaces conv1 with per-marker (64, 1, 7×7) kernels generated by
# a single linear layer from learned marker embeddings. embed_dim is chosen so
# that the total stem parameter count matches the no_pca Conv2d(K, 64, 7×7).
#
# Parameter budget:
#   no_pca:    K × 64 × 49  params (direct kernel weights)
#   hyperconv: vocab × e + e × 64 × 49  params  where e = round(K×3136/(vocab+3136))
# → both stems have essentially the same capacity.
#
# In multi-task training, marker_ids is passed PER BATCH (not stored as a fixed
# buffer), so shared embeddings are updated by all 11 task configurations jointly.

class MarkerStem(nn.Module):
    """
    Input stem that generates Conv2d kernels from learned marker embeddings.
    Each of the K input channels gets its own (out_ch, 1, 7, 7) kernel via a
    single linear projection of its marker embedding; results are summed →
    (B, 64, H/2, W/2), same shape as ResNet50 conv1 output.
    """
    def __init__(self, marker_vocab_size: int, embed_dim: int, out_ch: int = 64):
        super().__init__()
        self.out_ch     = out_ch
        self.embeddings = nn.Embedding(marker_vocab_size, embed_dim)
        self.hypernet   = nn.Linear(embed_dim, out_ch * 7 * 7, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor, marker_ids: torch.Tensor) -> torch.Tensor:
        # x: (B, K, H, W),  marker_ids: (B, K) or (K,) — vocabulary indices
        if marker_ids.dim() == 1:
            marker_ids = marker_ids.unsqueeze(0).expand(x.shape[0], -1)
        # Use first sample's ids to generate kernels (all samples in a batch share the same task)
        ids     = marker_ids[0]                                        # (K,)
        K       = x.shape[1]
        embs    = self.embeddings(ids)                                 # (K, embed_dim)
        kernels = self.hypernet(embs).view(K, self.out_ch, 1, 7, 7)   # (K, 64, 1, 7, 7)
        out = sum(
            F.conv2d(x[:, k:k+1], kernels[k], padding=3, stride=2)
            for k in range(K)
        )                                                              # (B, 64, H/2, W/2)
        return self.relu(self.bn(out))


class ResNet50HyperEncoder(nn.Module):
    """ResNet50 encoder with conv1+bn1+relu replaced by MarkerStem."""
    def __init__(self, marker_vocab_size: int, embed_dim: int):
        super().__init__()
        backbone     = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.stem    = MarkerStem(marker_vocab_size, embed_dim, out_ch=64)
        self.maxpool = backbone.maxpool
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.layer4  = backbone.layer4
        for param in backbone.parameters():
            param.requires_grad = False
        for param in self.layer2.parameters():
            param.requires_grad = True
        for param in self.layer3.parameters():
            param.requires_grad = True
        for param in self.layer4.parameters():
            param.requires_grad = True

    def forward(self, x: torch.Tensor, marker_ids: torch.Tensor):
        f0      = self.stem(x, marker_ids)   # (B, 64, H/2, W/2)
        f0_pool = self.maxpool(f0)
        f1 = self.layer1(f0_pool)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return f0, f1, f2, f3, f4


class ResNetHyperUNet(nn.Module):
    """
    ResNetUNet with MarkerStem replacing the encoder's conv1+bn1+relu.
    In multi-task training, marker_ids is passed per forward call (varies per batch).
    The stored buffer is a fallback default only.
    """
    def __init__(self, marker_vocab_size: int, embed_dim: int, default_marker_ids: torch.Tensor):
        super().__init__()
        self.encoder = ResNet50HyperEncoder(marker_vocab_size, embed_dim)
        self.decoder = UNetDecoder()
        self.register_buffer('marker_ids', default_marker_ids)

    def forward(self, x: torch.Tensor, marker_ids: torch.Tensor = None) -> torch.Tensor:
        ids = marker_ids if marker_ids is not None else self.marker_ids
        return self.decoder(self.encoder(x, ids))

def to_hwc(img):
    if img.ndim == 2:
        img = img[..., None]
    if img.shape[0] < img.shape[1] and img.shape[0] < img.shape[2]:
        img = np.transpose(img, (1, 2, 0))
    return img.astype(np.float32, copy=False)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fit_channel_preprocessing(train_files, pred_channel_indices, control_indices,
                               cofactor=5.0, eps=1e-8, sample_per_file=50_000):
    """
    Fit per-channel preprocessing shared across all marker-prediction tasks.

    Returns:
      channel_preproc:   {ch_idx: (reg_model, float mean, float std)}
                         Residual z-score stats for input channels.
      target_norm_stats: {ch_idx: (float p01, float p99)}
                         Training-set 1st/99th arcsinh percentiles for target
                         normalisation.  Using global stats preserves absolute
                         intensity differences across images (per-image min-max
                         would erase them).
    """
    if not train_files:
        raise ValueError(
            "fit_channel_preprocessing: train_files is empty.\n"
            "Check that the dataset path is correct and contains .tif/.tiff files.\n"
            "On Kaggle, files may live in a subdirectory — the glob in cell 13 now "
            "searches recursively, so re-run that cell first."
        )

    reg_models = {j: SGDRegressor(max_iter=1000, tol=1e-3) for j in pred_channel_indices}
    n_ctrl = len(control_indices)

    # Pass 1: fit regression (control markers → prediction channel)
    for c, f in enumerate(train_files):
        if c % max(1, len(train_files) // 4) == 0 or c == len(train_files) - 1:
            print(f'[prepro] regression pass: {c+1}/{len(train_files)}')
        img = to_hwc(tifffile.imread(f))
        Xc  = img[..., control_indices].reshape(-1, n_ctrl)
        for j in pred_channel_indices:
            reg_models[j].partial_fit(Xc, img[..., j].reshape(-1))
        del img, Xc; gc.collect()

    # Pass 2: residual arcsinh z-score stats + raw arcsinh samples for percentiles
    sums        = {j: 0.  for j in pred_channel_indices}
    sq          = {j: 0.  for j in pred_channel_indices}
    cnt         = {j: 0   for j in pred_channel_indices}
    arc_samples = {j: []  for j in pred_channel_indices}

    for c, f in enumerate(train_files):
        if c % max(1, len(train_files) // 4) == 0 or c == len(train_files) - 1:
            print(f'[prepro] stats pass: {c+1}/{len(train_files)}')
        img = to_hwc(tifffile.imread(f))
        Xc  = img[..., control_indices].reshape(-1, n_ctrl)
        N   = Xc.shape[0]
        for j in pred_channel_indices:
            raw = img[..., j].reshape(-1)
            r = np.arcsinh((raw - reg_models[j].predict(Xc)) / cofactor)
            sums[j] += r.sum(); sq[j] += (r * r).sum(); cnt[j] += N
            raw_arc = np.arcsinh(raw / cofactor)
            idx = np.random.choice(N, min(sample_per_file, N), replace=False)
            arc_samples[j].append(raw_arc[idx])
        del img, Xc; gc.collect()

    channel_preproc   = {}
    target_norm_stats = {}
    for j in pred_channel_indices:
        n = cnt[j]
        if n == 0:
            raise RuntimeError(
                f"fit_channel_preprocessing: channel {j} has 0 accumulated pixels — "
                "this should not happen if train_files is non-empty."
            )
        m = sums[j] / n
        s = float(np.sqrt(max(sq[j] / n - m ** 2, 0))) + eps
        channel_preproc[j] = (reg_models[j], float(m), s)
        all_arc = np.concatenate(arc_samples[j])
        target_norm_stats[j] = (float(np.percentile(all_arc, 1)),
                                float(np.percentile(all_arc, 99)))

    print(f'[prepro] fitted {len(channel_preproc)} channels; '
          f'target ranges: { {j: (f"{p01:.2f}", f"{p99:.2f}") for j, (p01,p99) in target_norm_stats.items()} }')
    return channel_preproc, target_norm_stats


def preprocess_multi(file_path, target_idx, input_channels, channel_preproc,
                     target_norm_stats, control_indices, cofactor=5.0, index_map=None):
    """
    Load one image and return:
      x: (K, H, W) — arcsinh-z-scored residuals for input_channels
      y: (1, H, W) — arcsinh globally normalised target, clipped to [0, 1]

    target_idx and keys in target_norm_stats are Jackson-space indices.
    index_map remaps them to the actual channel positions in file_path (used for
    Danenberg test images where channel order differs from Jackson training data).
    """
    img = to_hwc(tifffile.imread(file_path))
    H, W, _ = img.shape

    def _r(i):
        return index_map[i] if index_map is not None else i

    # Target: arcsinh → global percentile normalisation → clip [0,1]
    y_arc = np.arcsinh(img[..., _r(target_idx)] / cofactor).astype(np.float32)
    p01, p99 = target_norm_stats[target_idx]
    y = torch.from_numpy(
        np.clip((y_arc - p01) / (p99 - p01 + 1e-8), 0., 1.)
    ).unsqueeze(0).float()

    # Inputs: regression residuals → arcsinh → channel z-score
    n_ctrl = len(control_indices)
    Xc     = img[..., [_r(i) for i in control_indices]].reshape(-1, n_ctrl)
    chans  = []
    for j in input_channels:
        reg, m, s = channel_preproc[j]
        r = np.arcsinh((img[..., _r(j)].reshape(-1) - reg.predict(Xc)) / cofactor)
        chans.append(((r - m) / (s + 1e-8)).reshape(H, W).astype(np.float32))

    x = torch.from_numpy(np.stack(chans, 0)).float()
    return x, y


def augment_pair(x: torch.Tensor, y: torch.Tensor):
    """
    Apply an identical random spatial transform to input x (K,H,W) and target y (1,H,W).
    Flips, 90° rotations, and random crop — no intensity changes so z-scored residuals
    stay calibrated. Pure torch ops, no new dependencies.
    """
    K  = x.shape[0]
    xy = torch.cat([x, y], dim=0)          # (K+1, H, W) — joint spatial transform

    if random.random() < 0.5:
        xy = xy.flip(-1)                   # horizontal flip
    if random.random() < 0.5:
        xy = xy.flip(-2)                   # vertical flip

    k = random.randint(0, 3)
    if k:
        xy = torch.rot90(xy, k, [-2, -1])  # 90° / 180° / 270° rotation

    _, H, W = xy.shape
    ch   = random.randint(max(64, int(H * 0.75)), H)
    cw   = random.randint(max(64, int(W * 0.75)), W)
    top  = random.randint(0, H - ch)
    left = random.randint(0, W - cw)
    xy   = xy[:, top:top+ch, left:left+cw]

    return xy[:K], xy[K:]


def pad_collate(batch):
    X_list, Y_list, ids_list, orig = zip(*batch)
    maxH = max(x.shape[1] for x in X_list)
    maxW = max(x.shape[2] for x in X_list)
    Xp  = torch.stack([F.pad(x, (0, maxW-x.shape[2], 0, maxH-x.shape[1])) for x in X_list])
    Yp  = torch.stack([F.pad(y, (0, maxW-y.shape[2], 0, maxH-y.shape[1])) for y in Y_list])
    ids = torch.stack(ids_list)   # (B, K)
    return Xp, Yp, ids, orig

# ─── Preprocessing ablation arms (b) gamma  and  (c) ImmuVis-style ────────────
#
# Same fit/transform interface as fit_channel_preprocessing / preprocess_multi so
# they drop into MultiTaskDataset unchanged. The TARGET normalisation is identical
# to preprocess_multi in every arm (reuses target_norm_stats) — only the INPUT
# channel representation changes, keeping Pearson AND FID comparable across arms.


# ===== (b) gamma — scTransform-inspired multiplicative correction =============

def fit_gamma_preprocessing(train_files, pred_channel_indices, control_indices,
                            cofactor=5.0, eps=1e-8, sample_per_file=50_000,
                            n_median_sample=2_000_000):
    """
    Differs from fit_channel_preprocessing in three ways (session_summary P1.b):
      * multiplicative / log-space: operates on arcsinh(x), not raw intensity;
      * single anchored technical axis  T = PC1(arcsinh(control channels))  instead
        of a per-channel multivariate control regression;
      * median-centering: subtract gamma_m * (T - median T), so typical-permeability
        pixels (T = median T) are left unchanged.

    Returns channel_preproc {ch_idx: (pca, gamma_m, median_T, mean, std)}.
    The fitted 1-component IncrementalPCA is shared and stored in every tuple.

    gamma_m and the z-score stats are derived analytically from streaming first/second
    moments (median-centering shifts T by a constant, which changes neither cov nor var):
        gamma_m   = cov(arcsinh x_m, T) / var(T)
        var(corr) = var(arcsinh x_m) - gamma_m^2 * var(T)
    """
    n_ctrl = len(control_indices)

    # Pass 1: fit 1-component PCA on arcsinh(control channels) -> technical axis T
    pca = IncrementalPCA(n_components=1)
    for c, f in enumerate(train_files):
        if c % max(1, len(train_files) // 4) == 0 or c == len(train_files) - 1:
            print(f'[gamma] pca pass: {c+1}/{len(train_files)}')
        img = to_hwc(tifffile.imread(f))
        Xc  = np.arcsinh(img[..., control_indices].reshape(-1, n_ctrl) / cofactor)
        N   = Xc.shape[0]
        idx = np.random.choice(N, min(sample_per_file, N), replace=False)
        pca.partial_fit(Xc[idx])
        del img, Xc; gc.collect()

    # Pass 2: streaming moments of T and arcsinh(marker); subsample T for the median
    sum_T = sum_T2 = n_T = 0.0
    sum_x  = {j: 0. for j in pred_channel_indices}
    sum_x2 = {j: 0. for j in pred_channel_indices}
    sum_xT = {j: 0. for j in pred_channel_indices}
    T_samples = []
    for c, f in enumerate(train_files):
        if c % max(1, len(train_files) // 4) == 0 or c == len(train_files) - 1:
            print(f'[gamma] moments pass: {c+1}/{len(train_files)}')
        img = to_hwc(tifffile.imread(f))
        Xc  = np.arcsinh(img[..., control_indices].reshape(-1, n_ctrl) / cofactor)
        T   = pca.transform(Xc).reshape(-1)
        N   = T.shape[0]
        sum_T += T.sum(); sum_T2 += float((T * T).sum()); n_T += N
        idx = np.random.choice(N, min(sample_per_file, N), replace=False)
        T_samples.append(T[idx])
        for j in pred_channel_indices:
            ax = np.arcsinh(img[..., j].reshape(-1) / cofactor)
            sum_x[j]  += ax.sum()
            sum_x2[j] += float((ax * ax).sum())
            sum_xT[j] += float((ax * T).sum())
        del img, Xc, T; gc.collect()

    mean_T = sum_T / n_T
    var_T  = max(sum_T2 / n_T - mean_T ** 2, eps)
    T_all  = np.concatenate(T_samples)
    if T_all.shape[0] > n_median_sample:
        T_all = np.random.choice(T_all, n_median_sample, replace=False)
    median_T = float(np.median(T_all))

    channel_preproc = {}
    for j in pred_channel_indices:
        mean_x = sum_x[j] / n_T
        cov_xT = sum_xT[j] / n_T - mean_x * mean_T
        gamma  = cov_xT / var_T
        mean_c = mean_x - gamma * (mean_T - median_T)
        var_c  = max(sum_x2[j] / n_T - mean_x ** 2 - gamma ** 2 * var_T, eps)
        std_c  = float(np.sqrt(var_c)) + eps
        channel_preproc[j] = (pca, float(gamma), float(median_T), float(mean_c), std_c)

    print(f'[gamma] fitted {len(channel_preproc)} channels; '
          f'median_T={median_T:.3f} var_T={var_T:.3f} '
          f'gamma range=[{min(v[1] for v in channel_preproc.values()):.3f}, '
          f'{max(v[1] for v in channel_preproc.values()):.3f}]')
    return channel_preproc


def preprocess_gamma(file_path, target_idx, input_channels, channel_preproc,
                     target_norm_stats, control_indices, cofactor=5.0, index_map=None):
    img = to_hwc(tifffile.imread(file_path))
    H, W, _ = img.shape

    def _r(i):
        return index_map[i] if index_map is not None else i

    # Target: identical to preprocess_multi (shared across all arms)
    y_arc = np.arcsinh(img[..., _r(target_idx)] / cofactor).astype(np.float32)
    p01, p99 = target_norm_stats[target_idx]
    y = torch.from_numpy(
        np.clip((y_arc - p01) / (p99 - p01 + 1e-8), 0., 1.)
    ).unsqueeze(0).float()

    # Inputs: arcsinh -> subtract gamma*(T - median_T) -> channel z-score
    pca    = next(iter(channel_preproc.values()))[0]
    n_ctrl = len(control_indices)
    Xc = np.arcsinh(img[..., [_r(i) for i in control_indices]].reshape(-1, n_ctrl) / cofactor)
    T  = pca.transform(Xc).reshape(H, W)
    chans = []
    for j in input_channels:
        _, gamma, median_T, m, s = channel_preproc[j]
        ax        = np.arcsinh(img[..., _r(j)] / cofactor)
        corrected = ax - gamma * (T - median_T)
        chans.append(((corrected - m) / (s + 1e-8)).astype(np.float32))
    x = torch.from_numpy(np.stack(chans, 0)).float()
    return x, y


# ===== (c) ImmuVis-style — arcsinh + Butterworth low-pass + percentile min-max =

def _butter2d(img2d, order=2, cutoff=0.2):
    """2D Butterworth low-pass (separable, zero-phase) — Uchal2026 App. A.7."""
    sos = butter(order, cutoff, btype='low', output='sos')
    out = sosfiltfilt(sos, img2d, axis=0)
    out = sosfiltfilt(sos, out,  axis=1)
    return np.ascontiguousarray(out, dtype=np.float32)


def fit_immuvis_preprocessing(train_files, pred_channel_indices, control_indices,
                              cofactor=5.0, eps=1e-8, sample_per_file=50_000,
                              order=2, cutoff=0.2):
    """
    Signal-processing baseline (session_summary P1.c): arcsinh(x/5) -> 2D Butterworth
    low-pass (order 2, 0.2 Nyquist) -> per-channel 1st/99th-percentile min-max -> clip
    [0,1].  NO structural-control regression.  control_indices is accepted only for a
    common signature with the other arms and is unused here.

    Returns channel_preproc {ch_idx: (p01, p99, order, cutoff)}.
    """
    arc_samples = {j: [] for j in pred_channel_indices}
    for c, f in enumerate(train_files):
        if c % max(1, len(train_files) // 4) == 0 or c == len(train_files) - 1:
            print(f'[immuvis] stats pass: {c+1}/{len(train_files)}')
        img = to_hwc(tifffile.imread(f))
        for j in pred_channel_indices:
            s   = _butter2d(np.arcsinh(img[..., j] / cofactor), order, cutoff).reshape(-1)
            idx = np.random.choice(s.shape[0], min(sample_per_file, s.shape[0]), replace=False)
            arc_samples[j].append(s[idx])
        del img; gc.collect()

    channel_preproc = {}
    for j in pred_channel_indices:
        alls = np.concatenate(arc_samples[j])
        channel_preproc[j] = (float(np.percentile(alls, 1)),
                              float(np.percentile(alls, 99)),
                              int(order), float(cutoff))
    print(f'[immuvis] fitted {len(channel_preproc)} channels (order={order}, cutoff={cutoff})')
    return channel_preproc


def preprocess_immuvis(file_path, target_idx, input_channels, channel_preproc,
                       target_norm_stats, control_indices, cofactor=5.0, index_map=None):
    img = to_hwc(tifffile.imread(file_path))
    H, W, _ = img.shape

    def _r(i):
        return index_map[i] if index_map is not None else i

    # Target: identical to preprocess_multi (shared across all arms)
    y_arc = np.arcsinh(img[..., _r(target_idx)] / cofactor).astype(np.float32)
    p01, p99 = target_norm_stats[target_idx]
    y = torch.from_numpy(
        np.clip((y_arc - p01) / (p99 - p01 + 1e-8), 0., 1.)
    ).unsqueeze(0).float()

    # Inputs: arcsinh -> Butterworth low-pass -> percentile min-max -> clip [0,1]
    chans = []
    for j in input_channels:
        lo, hi, order, cutoff = channel_preproc[j]
        s = _butter2d(np.arcsinh(img[..., _r(j)] / cofactor), order, cutoff)
        chans.append(np.clip((s - lo) / (hi - lo + 1e-8), 0., 1.).astype(np.float32))
    x = torch.from_numpy(np.stack(chans, 0)).float()
    return x, y


# ─── Dataset ──────────────────────────────────────────────────────────────────

class MultiTaskDataset(Dataset):
    """
    Expands to len(file_list) × len(tasks) items.  Each item is a (file, task) pair
    that yields (x, y, marker_ids, (H, W)) where:
      x:          (K, H, W)  arcsinh-z-scored input channels
      y:          (1, H, W)  arcsinh globally-normalised target (no per-image rescaling)
      marker_ids: (K,)       Jackson vocabulary indices of the K input channels
      (H, W):                original spatial size before any padding

    tasks: list of (target_idx, input_channels_sorted) tuples — one per marker.
    Input channels are sorted by Jackson index so no_pca channel positions are
    consistent across tasks.

    Val/test datasets are constructed with augment=False.
    File-level train/val split is handled externally via Subset on file indices.
    """
    def __init__(self, file_list, tasks, channel_preproc, target_norm_stats,
                 control_indices, cofactor=5.0, augment=False, index_map=None,
                 preproc='residual'):
        self.file_list        = file_list
        self.tasks            = tasks
        self.channel_preproc  = channel_preproc
        self.target_norm_stats = target_norm_stats
        self.control_indices  = control_indices
        self.cofactor         = cofactor
        self.augment          = augment
        self.index_map        = index_map
        self.preproc          = preproc
        self.items = [(fi, ti)
                      for fi in range(len(file_list))
                      for ti in range(len(tasks))]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        fi, ti = self.items[idx]
        target_idx, input_channels = self.tasks[ti]
        _prep_fn = {
            'residual': preprocess_multi,
            'gamma':    preprocess_gamma,
            'immuvis':  preprocess_immuvis,
        }[self.preproc]
        x, y = _prep_fn(
            self.file_list[fi], target_idx, input_channels,
            self.channel_preproc, self.target_norm_stats,
            self.control_indices, self.cofactor, self.index_map,
        )
        if self.augment:
            x, y = augment_pair(x, y)
        return x, y, torch.tensor(input_channels, dtype=torch.long), (x.shape[1], x.shape[2])

def smart_load_state_dict(model, state, strict=True):
    is_dp      = isinstance(model, nn.DataParallel)
    has_module = all(k.startswith('module.') for k in state.keys())
    if is_dp and not has_module:
        return model.module.load_state_dict(state, strict=strict)
    if not is_dp and has_module:
        state = {k.replace('module.', '', 1): v for k, v in state.items()}
    return (model.module if is_dp else model).load_state_dict(state, strict=strict)

In [ ]:
j_idx = {m: i for i, m in enumerate(J_CHAN)}
d_idx = {m: i for i, m in enumerate(D_CHAN)}
shared = sorted(set(J_CHAN) & set(D_CHAN))
control_train = [j_idx[m] for m in NUC]
train_to_test_index_map = {j_idx[m]: d_idx[m] for m in shared}
all_tasks = [(j_idx[m], sorted([j_idx[mm] for mm in MARKERS if mm != m])) for m in MARKERS]
task_names = list(MARKERS)
CACHE = os.path.dirname(g("channel_preproc_residual.pkl")[0])
channel_preproc_residual, target_norm_stats = pickle.load(open(f"{CACHE}/channel_preproc_residual.pkl", "rb"))
CPREP = {"residual": channel_preproc_residual,
         "gamma": pickle.load(open(f"{CACHE}/channel_preproc_gamma.pkl", "rb")),
         "immuvis": pickle.load(open(f"{CACHE}/channel_preproc_immuvis.pkl", "rb"))}
CKPT = {a: f"{CACHE}/model_jackson2020_to_danenberg2022_ablate_{a}_full_nopca_augFalse.pth" for a in ARMS}
assert all(os.path.exists(p) for p in CKPT.values()), CKPT
device = "cuda" if torch.cuda.is_available() else "cpu"
print("cache:", CACHE, "| device:", device, "| tasks:", task_names)

## Images, masks and the pixel → cell map

In [ ]:
tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
IMGS = list(DD["img_names"])
missing = [s for s in IMGS if s not in D_IMG or s not in D_MASK]
assert not missing, f"{len(missing)} images without a tiff or mask, e.g. {missing[:5]}"
FILES = [D_IMG[s] for s in IMGS]
OBJ = DD["cells"].object_number.values.astype(np.int64)

def pixmap(mask_path, obj):
    """Small arguments only: joblib workers must not receive the big tables."""
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    return v.astype(np.int32), loc[v].astype(np.int32), len(obj), tuple(m.shape)

ST = DD["starts"]
PIXMAP = Parallel(n_jobs=os.cpu_count())(delayed(pixmap)(D_MASK[s], OBJ[a:b])
                                         for s, a, b in zip(IMGS, ST[:-1], ST[1:]))
cover = np.array([np.unique(pm[1]).size / pm[2] for pm in PIXMAP])
print(f"cells with ≥1 mask pixel: median {np.median(cover):.4f}, min {cover.min():.4f}")
tick("pixel maps")

def cell_mean(flat, pm):
    v, loc, n, _ = pm
    cnt = np.bincount(loc, minlength=n)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnt > 0, np.bincount(loc, flat[v], n) / np.maximum(cnt, 1), np.nan).astype(np.float32)

def raw_means(path, pm):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    return np.stack([cell_mean(im[:, c], pm) for c in range(im.shape[1])], 1)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(f, pm) for f, pm in zip(FILES, PIXMAP)))
assert RAW.shape == (N, len(D_CHAN)), RAW.shape
for c in ("histone_h3", "dna1", "pan_cytokeratin", "estrogen_receptor_alpha"):
    tc = D_TABLE.get(c, c)
    print(f"image-derived vs published table, {c}: r = {np.corrcoef(np.nan_to_num(RAW[:, d_idx[c]]), DD['cells'][tc].values)[0, 1]:.3f}")
tick("raw per-cell means")

## GPU pass: per-cell truth, prediction and shifted predictions for every (arm, marker)

In [ ]:
TRUTH, TSHIFT, PRED, SHIFT, PIX = {}, {}, {}, {}, []
for arm in ARMS:
    model = ResNetUNet(in_channels=len(all_tasks[0][1])).to(device)
    smart_load_state_dict(model, torch.load(CKPT[arm], map_location=device))
    model.eval()
    for m in MARKERS:
        tgt, inp = all_tasks[task_names.index(m)]
        ds = MultiTaskDataset(FILES, [(tgt, inp)], CPREP[arm], target_norm_stats, control_train, cofactor=5.0,
                              augment=False, index_map=train_to_test_index_map, preproc=arm)
        loader = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=pad_collate, num_workers=2, pin_memory=True)
        pred = np.full(N, np.nan, np.float32)
        shf = np.full((N, N_SHIFT), np.nan, np.float32)
        need = m not in TRUTH
        if need:
            tru, tsh = np.full(N, np.nan, np.float32), np.full(N, np.nan, np.float32)
        i = 0
        with torch.no_grad():
            for x, y, mids, sizes in loader:
                x, mids = x.to(device, non_blocking=True), mids.to(device, non_blocking=True)
                with autocast(device_type="cuda", enabled=(device == "cuda")):
                    yh = model(x, mids)
                yh = yh.float().clamp(0, 1).cpu().numpy()
                yn = y.numpy()
                for b, (h, w) in enumerate(sizes):
                    pm = PIXMAP[i]
                    assert pm[3] == (h, w), (IMGS[i], pm[3], (h, w))
                    a0, n_i = DD["starts"][i], pm[2]
                    Pm, Gm = yh[b, 0, :h, :w], yn[b, 0, :h, :w]
                    pred[a0:a0 + n_i] = cell_mean(Pm.ravel(), pm)
                    rr = np.random.default_rng(SEED + i)
                    offs = [(int(rr.integers(h // 4, 3 * h // 4 + 1)), int(rr.integers(w // 4, 3 * w // 4 + 1)))
                            for _ in range(N_SHIFT)]
                    for k, (dy, dx) in enumerate(offs):
                        shf[a0:a0 + n_i, k] = cell_mean(np.roll(Pm, (dy, dx), (0, 1)).ravel(), pm)
                    if need:
                        tru[a0:a0 + n_i] = cell_mean(Gm.ravel(), pm)
                        tsh[a0:a0 + n_i] = cell_mean(np.roll(Gm, offs[0], (0, 1)).ravel(), pm)
                    gf, pf = Gm.ravel().astype(np.float64), Pm.ravel().astype(np.float64)
                    PIX.append(dict(arm=arm, marker=m, image=IMGS[i], patient=DD["patient"][a0], n=gf.size,
                                    Sg=gf.sum(), Sp=pf.sum(), Sgg=gf @ gf, Spp=pf @ pf, Sgp=gf @ pf))
                    i += 1
        assert i == len(FILES), (i, len(FILES))
        PRED[(arm, m)], SHIFT[(arm, m)] = pred, shf
        if need:
            TRUTH[m], TSHIFT[m] = tru, tsh
        tick(f"{arm} {m}")
    del model
    torch.cuda.empty_cache()
PIX = pd.DataFrame(PIX)
PIX.to_csv(f"{OUT}/pixel_sums_per_image.csv", index=False)
cols = {f"truth__{m}": TRUTH[m] for m in MARKERS}
cols.update({f"pred__{a}__{m}": PRED[(a, m)] for a in ARMS for m in MARKERS})
cols.update({f"shift1__{a}__{m}": SHIFT[(a, m)][:, 0] for a in ARMS for m in MARKERS})
pd.concat([DD["cells"][["image_name", "object_number"]].reset_index(drop=True),
           pd.DataFrame(cols).astype(np.float16)], axis=1).to_parquet(f"{OUT}/percell_truth_pred.parquet", index=False)
tick("GPU pass done; per-cell table saved")

## Cell-table baselines (trained on the Jackson table, applied to Danenberg image-derived cells)

In [ ]:
try:
    def harm(X):
        med = np.nanmedian(X, 0)
        q05, q95 = np.nanpercentile(X, [5, 95], 0)
        sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
        return ((X - med) / sc).astype(np.float32)

    def rank_norm(v):
        return pd.Series(v).rank(pct=True).values.astype(np.float32)

    dix = {m: i for i, m in enumerate(D_CHAN)}
    IN_SET = NUC + MARKERS                                   # 14 channels, same names in both cohorts
    XD_in = np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF).astype(np.float32)
    XJ_in = np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF).astype(np.float32)
    HJ, HD = harm(XJ_in), harm(XD_in)
    RJ = np.column_stack([rank_norm(XJ_in[:, j]) for j in range(XJ_in.shape[1])])
    RD = np.column_stack([rank_norm(XD_in[:, j]) for j in range(XD_in.shape[1])])
    CJ, CDm = contact_matrix(DJ), contact_matrix(DD)
    MrJ, nrJ = ring_means(HJ, DJ, CJ)
    MrD, nrD = ring_means(HD, DD, CDm)
    aJ, aD = DJ["log_area"][:, None], DD["log_area"][:, None]
    tick("baseline features")

    def fitpred(Xtr, ytr, Xte):
        mdl = lgb.LGBMRegressor(random_state=SEED, **LGB)
        mdl.fit(Xtr, ytr)
        return mdl.predict(Xte).astype(np.float32)

    BASE = {}
    trJ = np.sort(np.random.default_rng(SEED).choice(len(XJ_in), min(N_FIT_FINAL, len(XJ_in)), replace=False))
    for m in MARKERS:
        k = IN_SET.index(m)
        cols = [i for i in range(len(IN_SET)) if i != k]
        yJ = XJ_in[trJ, k]
        BASE[("B1", m)] = fitpred(np.c_[HJ[trJ][:, cols], aJ[trJ]], yJ, np.c_[HD[:, cols], aD])
        BASE[("B1r", m)] = fitpred(np.c_[RJ[trJ][:, cols], aJ[trJ]], yJ, np.c_[RD[:, cols], aD])
        BASE[("B2", m)] = fitpred(np.c_[HJ[trJ][:, cols], aJ[trJ], MrJ[trJ][:, cols], nrJ[trJ, None]], yJ,
                                  np.c_[HD[:, cols], aD, MrD[:, cols], nrD[:, None]])
    tick("baselines fitted")
except Exception:
    print('!!! block baselines failed: ' + traceback.format_exc(), flush=True)
    FAILED.append('baselines')

## Rungs: pooled pixel r, R4, R5 (with nulls and positive controls)

In [ ]:
try:
    IMG = DD["img"]
    NP_ = len(pats)

    def within_stats(a, b):
        ok = np.isfinite(a) & np.isfinite(b)
        a, b, im, pc = a[ok].astype(np.float64), b[ok].astype(np.float64), IMG[ok], pcode[ok]
        k = int(IMG.max()) + 1
        cnt = np.maximum(np.bincount(im, minlength=k), 1)
        ac = a - (np.bincount(im, a, k) / cnt)[im]
        bc = b - (np.bincount(im, b, k) / cnt)[im]
        return np.bincount(pc, ac * bc, NP_), np.bincount(pc, ac * ac, NP_), np.bincount(pc, bc * bc, NP_)

    def corr_boot(st):
        sxy, sxx, syy = st
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))

    def ci(v):
        v = np.asarray(v)[np.isfinite(v)]
        return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)

    half = (np.random.default_rng(SEED + 5).random(NP_) < 0.5)[pcode]

    def crossfit(v, X, seed):
        """Residual v − f(X); f is fitted on the other half of the patients (state = what X implies)."""
        out = np.full(len(v), np.nan, np.float32)
        rng = np.random.default_rng(seed)
        for h in (False, True):
            tr = np.flatnonzero((half == h) & np.isfinite(v))
            te = np.flatnonzero(half != h)
            tr = np.sort(rng.choice(tr, min(N_FIT, len(tr)), replace=False))
            mdl = lgb.LGBMRegressor(random_state=SEED, **LGB).fit(X[tr], v[tr])
            out[te] = v[te] - mdl.predict(X[te])
        return out

    def shuffle_within(v, key, seed):
        r = np.random.default_rng(seed).random(len(v))
        s1 = np.lexsort((np.arange(len(v)), key))
        s2 = np.lexsort((r, key))
        out = np.empty_like(v)
        out[s1] = v[s2]
        return out

    ROWS, BOOT = [], {}

    def record(m, src, rung, point, bvec):
        BOOT[(m, src, rung)] = (point, bvec)
        lo, hi = ci(bvec)
        ROWS.append(dict(marker=m, source=src, rung=rung, value=point, lo=lo, hi=hi))

    PIXP = PIX.groupby(["arm", "marker", "patient"])[["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]].sum()

    def r_from(n, Sg, Sp, Sgg, Spp, Sgp):
        return (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))

    CONTROL = {}
    for mi, m in enumerate(MARKERS):
        tru = TRUTH[m]
        k = IN_SET.index(m)
        cols = [i for i in range(len(IN_SET)) if i != k]
        X_in = np.c_[HD[:, cols], aD]
        X_pan = np.c_[np.arcsinh(RAW[:, [i for i in range(len(D_CHAN)) if i != dix[m]]] / COF), aD]
        src = {a: PRED[(a, m)] for a in ARMS}
        src.update({f"{a} shifted": SHIFT[(a, m)][:, 0] for a in ARMS})
        src.update({b: BASE[(b, m)] for b in ("B1", "B1r", "B2")})
        # pooled pixel r (arms)
        for a in ARMS:
            t = PIXP.loc[(a, m)].reindex(pats).fillna(0)
            cols6 = ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]
            record(m, a, "pooled", float(r_from(*(t[c].values.sum() for c in cols6))),
                   r_from(*(W @ t[c].values for c in cols6)))
        # R4
        for s, v in src.items():
            record(m, s, "R4", *corr_boot(within_stats(tru, v)))
        for a in ARMS:      # R4 null averaged over the 5 shifts
            pts, bvs = zip(*[corr_boot(within_stats(tru, SHIFT[(a, m)][:, j])) for j in range(N_SHIFT)])
            record(m, f"{a} shifted (mean of 5)", "R4", float(np.mean(pts)), np.mean(bvs, 0))
        # R5 (cross-fitted residuals), input space and panel space
        seed = SEED + 100 * mi
        rt_in = crossfit(tru, X_in, seed)
        rt_pan = crossfit(tru, X_pan, seed + 1)
        state = tru - rt_in                                   # what the inputs imply about the truth
        dec = pd.qcut(pd.Series(state).rank(method="first"), 10, labels=False).fillna(-1).values.astype(np.int64)
        ctrl = {"control: shifted true map": TSHIFT[m], "control: state mean": state,
                "control: shuffled within state": shuffle_within(tru, IMG.astype(np.int64) * 11 + dec + 1, seed + 2),
                "control: identity": tru}
        CONTROL[m] = ctrl
        for s, v in ctrl.items():
            record(m, s, "R4", *corr_boot(within_stats(tru, v)))
        for s, v in list(src.items()) + [(s, v) for s, v in ctrl.items() if s != "control: identity"]:
            record(m, s, "R5_input", *corr_boot(within_stats(rt_in, crossfit(v, X_in, seed + 3))))
        record(m, "control: identity", "R5_input", *corr_boot(within_stats(rt_in, rt_in)))
        for s in ARMS + ["B1", "B2"]:
            record(m, s, "R5_panel", *corr_boot(within_stats(rt_pan, crossfit(src[s], X_pan, seed + 4))))
        tick(f"rungs {m}")
    RUNG = pd.DataFrame(ROWS)
    RUNG.to_csv(f"{OUT}/rungs.csv", index=False)
    print(RUNG.pivot_table(index=["marker", "source"], columns="rung", values="value").round(3).to_string())

    def diff(m, s1, s2, rung):
        p1, b1 = BOOT[(m, s1, rung)]
        p2, b2 = BOOT[(m, s2, rung)]
        lo, hi = ci(b1 - b2)
        return dict(marker=m, rung=rung, a=s1, b=s2, diff=p1 - p2, lo=lo, hi=hi)

    DIFF = []
    for m in MARKERS:
        for a in ARMS:
            for b in ("B1", "B2", "B1r"):
                DIFF += [diff(m, a, b, "R4"), diff(m, a, b, "R5_input")]
            DIFF.append(diff(m, a, f"{a} shifted", "R5_input"))
            DIFF.append(diff(m, a, f"{a} shifted (mean of 5)", "R4"))
        for i, a in enumerate(ARMS):
            for b in ARMS[i + 1:]:
                DIFF += [diff(m, a, b, "pooled"), diff(m, a, b, "R4")]
    DIFF = pd.DataFrame(DIFF)
    DIFF.to_csv(f"{OUT}/rung_differences.csv", index=False)
    cols = {f"{b}__{m}": BASE[(b, m)] for b in ("B1", "B1r", "B2") for m in MARKERS}
    pd.concat([DD["cells"][["image_name", "object_number"]].reset_index(drop=True),
               pd.DataFrame(cols).astype(np.float16)], axis=1).to_parquet(f"{OUT}/percell_baselines.parquet", index=False)
    tick("rungs done")
except Exception:
    print('!!! block rungs failed: ' + traceback.format_exc(), flush=True)
    FAILED.append('rungs')

## Downstream readouts (D3)

In [ ]:
TV = np.isin(DD["split"], ["train", "val"])
TRN, VAL = DD["split"] == "train", DD["split"] == "val"
EPI = DD["epi"]

def sources_for(m):
    s = {"truth": TRUTH[m]}
    s.update({a: PRED[(a, m)] for a in ARMS})
    s.update({f"{a} shifted": SHIFT[(a, m)][:, 0] for a in ARMS})
    s.update({b: BASE[(b, m)] for b in ("B1", "B2")})
    s["control: state mean"] = CONTROL[m]["control: state mean"]
    return s

READ = {"status": [], "f1": [], "mplus": []}

@guard("status")
def _():
    rng = np.random.default_rng(SEED + 7)
    for m, col in (("estrogen_receptor_alpha", "ER_STATUS"), ("c_erb_b_2_her2", "HER2_STATUS")):
        mask = TV & EPI
        pp = pd.DataFrame({k: pd.Series(v[mask]).groupby(DD["patient"][mask]).mean() for k, v in sources_for(m).items()})
        lab = clin[col].reindex(pp.index)
        ok = lab.isin(["Positive", "Negative"]).values & pp.notna().all(1).values
        pp, y = pp[ok], (lab[ok] == "Positive").astype(int).values
        boots = {k: [] for k in pp}
        for _ in range(N_BOOT):
            i = rng.integers(0, len(y), len(y))
            if y[i].min() == y[i].max():
                continue
            for kk in pp:
                boots[kk].append(roc_auc_score(y[i], pp[kk].values[i]))
        base = np.array(boots["truth"])
        for kk in pp:
            d = np.array(boots[kk]) - base
            READ["status"].append(dict(marker=m, status=col, source=kk, n=len(y), auc=roc_auc_score(y, pp[kk].values),
                                       delta_vs_truth=roc_auc_score(y, pp[kk].values) - roc_auc_score(y, pp["truth"].values),
                                       lo=ci(d)[0], hi=ci(d)[1]))
    pd.DataFrame(READ["status"]).to_csv(f"{OUT}/readout_status_auc.csv", index=False)
    print(pd.DataFrame(READ["status"]).round(3).to_string(index=False))

@guard("phenotype classifier")
def _():
    F = np.column_stack([rank_norm(RAW[:, j]) for j in range(RAW.shape[1])] + [DD["log_area"]])
    lab_codes, lab_names = pd.factorize(DD["phen"])
    tr = np.sort(np.random.default_rng(SEED + 8).choice(np.flatnonzero(TRN), min(2 * N_FIT, TRN.sum()), replace=False))
    va = np.flatnonzero(VAL)
    clf = lgb.LGBMClassifier(random_state=SEED, **{**LGB, "n_estimators": 30 if SMOKE else 150})
    clf.fit(F[tr], lab_codes[tr])
    C = len(lab_names)
    vp, _ = pd.factorize(DD["patient"][va])
    Wv = np.random.default_rng(SEED + 9).multinomial(vp.max() + 1, np.full(vp.max() + 1, 1 / (vp.max() + 1)),
                                                      size=N_BOOT_F1).astype(np.float64)

    def conf_by_patient(pred):
        M = np.zeros((vp.max() + 1, C, C))
        np.add.at(M, (vp, lab_codes[va], pred), 1)
        return M

    def macro_f1(Mc):
        tp = np.einsum("...ii->...i", Mc)
        fp, fn = Mc.sum(-2) - tp, Mc.sum(-1) - tp
        sup = Mc.sum(-1) > 0
        f1 = np.where(sup, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)
        return np.nanmean(f1, -1)

    Mm = conf_by_patient(clf.predict(F[va]))
    f_meas, b_meas = macro_f1(Mm.sum(0)), macro_f1(np.einsum("bp,pij->bij", Wv, Mm))
    for m in MARKERS:
        for s, v in sources_for(m).items():
            Fs = F[va].copy()
            Fs[:, dix[m]] = rank_norm(v)[va]
            Ms = conf_by_patient(clf.predict(Fs))
            d = macro_f1(np.einsum("bp,pij->bij", Wv, Ms)) - b_meas
            READ["f1"].append(dict(marker=m, source=s, f1_measured=f_meas, f1_substituted=macro_f1(Ms.sum(0)),
                                   delta=macro_f1(Ms.sum(0)) - f_meas, lo=ci(d)[0], hi=ci(d)[1]))
        tick(f"classifier substitution {m}")
    pd.DataFrame(READ["f1"]).to_csv(f"{OUT}/readout_phenotype_f1.csv", index=False)
    print(pd.DataFrame(READ["f1"]).pivot(index="marker", columns="source", values="delta").round(4).to_string())

@guard("M+ fraction")
def _():
    rng = np.random.default_rng(SEED + 10)
    va = np.flatnonzero(VAL)
    unit = pd.Series(DD["patient"][va]) + "|" + pd.Series(DD["broad"][va])
    for m in MARKERS:
        tru = TRUTH[m]
        pos_t = tru[va] > np.nanpercentile(tru[TRN], 90)
        for s, v in sources_for(m).items():
            if s == "truth":
                continue
            pos_s = v[va] > np.nanpercentile(v[TRN], 90)
            df = pd.DataFrame({"u": unit.values, "t": pos_t, "s": pos_s}).groupby("u").agg(t=("t", "mean"), s=("s", "mean"), n=("t", "size"))
            df = df[df.n >= 20]
            pu = df.index.str.split("|").str[0].values
            up, uc = np.unique(pu, return_inverse=True)
            r0 = np.corrcoef(df.t, df.s)[0, 1]
            bs = []
            for _ in range(N_BOOT_F1):
                pick = rng.integers(0, len(up), len(up))
                idx = np.concatenate([np.flatnonzero(uc == p) for p in pick])
                if df.t.values[idx].std() > 0 and df.s.values[idx].std() > 0:
                    bs.append(np.corrcoef(df.t.values[idx], df.s.values[idx])[0, 1])
            READ["mplus"].append(dict(marker=m, source=s, n_units=len(df), r=r0, lo=ci(bs)[0], hi=ci(bs)[1]))
    pd.DataFrame(READ["mplus"]).to_csv(f"{OUT}/readout_mplus_fraction.csv", index=False)
    print(pd.DataFrame(READ["mplus"]).pivot(index="marker", columns="source", values="r").round(3).to_string())
tick("readouts done")

## Pre-registered decisions

In [ ]:
DEC = {"failed_blocks": FAILED}

def ci_pos(df):
    return df[df.lo > 0]

@guard("decisions")
def _():
    d = DIFF
    beats = {a: {b: ci_pos(d[(d.rung == "R4") & (d.a == a) & (d.b == b)]).marker.tolist() for b in ("B1", "B2", "B1r")}
             for a in ARMS}
    DEC["image_beats_table_R4"] = beats
    DEC["image_beats_table"] = any(len(beats[a]["B1"]) >= 2 for a in ARMS)
    beyond = {a: ci_pos(d[(d.rung == "R5_input") & (d.a == a) & (d.b == f"{a} shifted")]).marker.tolist() for a in ARMS}
    DEC["beyond_panel_R5_vs_shift"] = beyond
    DEC["beyond_panel"] = len(set(sum(beyond.values(), []))) >= 2
    # D1: arm pairs ranked oppositely by pooled r and R4, both CIs excluding 0
    d1 = []
    for m in MARKERS:
        for i, a in enumerate(ARMS):
            for b in ARMS[i + 1:]:
                p = d[(d.marker == m) & (d.rung == "pooled") & (d.a == a) & (d.b == b)].iloc[0]
                r = d[(d.marker == m) & (d.rung == "R4") & (d.a == a) & (d.b == b)].iloc[0]
                sig = lambda x: (x.lo > 0) or (x.hi < 0)
                if sig(p) and sig(r) and np.sign(p["diff"]) != np.sign(r["diff"]):
                    d1.append(dict(marker=m, a=a, b=b, pooled_diff=p["diff"], r4_diff=r["diff"]))
    DEC["D1_pairs"] = d1
    DEC["D1"] = len({x["marker"] for x in d1}) >= 2
    # D2: LOMO R² of cell-level readouts, pooled r alone vs + R4 + R5_input (33 units)
    piv = RUNG[RUNG.source.isin(ARMS)].pivot_table(index=["marker", "source"], columns="rung", values="value")
    U = piv[["pooled", "R4", "R5_input"]].reset_index().rename(columns={"source": "arm"})
    if READ["f1"]:
        f1 = pd.DataFrame(READ["f1"]).rename(columns={"source": "arm", "delta": "f1_delta"})
        U = U.merge(f1[["marker", "arm", "f1_delta"]], on=["marker", "arm"], how="left")
    if READ["mplus"]:
        mp = pd.DataFrame(READ["mplus"]).rename(columns={"source": "arm", "r": "mplus_r"})
        U = U.merge(mp[["marker", "arm", "mplus_r"]], on=["marker", "arm"], how="left")
    U.to_csv(f"{OUT}/d2_units.csv", index=False)
    d2 = {}
    for y in [c for c in ("f1_delta", "mplus_r") if c in U]:
        res = {}
        Uy = U.dropna(subset=["pooled", "R4", "R5_input", y]).reset_index(drop=True)
        for feats in (["pooled"], ["pooled", "R4", "R5_input"]):
            pred = np.full(len(Uy), np.nan)
            for m in MARKERS:
                te = (Uy.marker == m).values
                if te.any():
                    lr = LinearRegression().fit(Uy.loc[~te, feats].values, Uy.loc[~te, y].values)
                    pred[te] = lr.predict(Uy.loc[te, feats].values)
            yy = Uy[y].values
            res["+".join(feats)] = float(1 - np.sum((yy - pred) ** 2) / np.sum((yy - yy.mean()) ** 2))
        res["gain"] = res["pooled+R4+R5_input"] - res["pooled"]
        d2[y] = res
    DEC["D2_lomo_r2"] = d2
    DEC["D2"] = any(v["gain"] >= 0.1 for v in d2.values())
    dis = DEC["D1"] or DEC["D2"]
    DEC["outcome"] = ("GO" if dis and DEC["beyond_panel"] else "PIVOT-A" if dis else
                      "PIVOT-B" if DEC["beyond_panel"] else "KILL")
    # positive controls: do they land on their rungs?
    rr = RUNG.set_index(["marker", "source", "rung"]).value
    pc = {}
    for m in MARKERS:
        pc[m] = dict(shifted_R4=rr[(m, "control: shifted true map", "R4")],
                     state_R4=rr[(m, "control: state mean", "R4")], state_R5=rr[(m, "control: state mean", "R5_input")],
                     shuffled_R4=rr[(m, "control: shuffled within state", "R4")],
                     shuffled_R5=rr[(m, "control: shuffled within state", "R5_input")])
    PC = pd.DataFrame(pc).T
    PC.to_csv(f"{OUT}/positive_controls.csv")
    DEC["controls_valid"] = dict(
        shifted_R4_below_0_05=int((PC.shifted_R4.abs() < 0.05).sum()),
        state_R5_below_0_05=int((PC.state_R5.abs() < 0.05).sum()),
        shuffled_R5_below_0_05=int((PC.shuffled_R5.abs() < 0.05).sum()),
        shuffled_R4_above_0_1=int((PC.shuffled_R4 > 0.1).sum()), n_markers=len(MARKERS))
    print(PC.round(3).to_string())

@guard("figures")
def _():
    col = {"B1": "#bdbdbd", "B2": "#737373", "residual": "#41ab5d", "gamma": "#2171b5", "immuvis": "#cb181d"}
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    x = np.arange(len(MARKERS))
    for ax, rung in zip(axes, ("R4", "R5_input", "R5_panel")):
        srcs = [s for s in ("B1", "B2", "residual", "gamma", "immuvis")
                if ((RUNG.rung == rung) & (RUNG.source == s)).any()]
        w = 0.8 / len(srcs)
        for j, s in enumerate(srcs):
            t = RUNG[(RUNG.rung == rung) & (RUNG.source == s)].set_index("marker").reindex(MARKERS)
            ax.bar(x + (j - (len(srcs) - 1) / 2) * w, t.value, w, color=col[s],
                   yerr=[t.value - t.lo, t.hi - t.value], capsize=1.5, error_kw=dict(lw=0.6), label=s)
        if rung != "R5_panel":
            nul = RUNG[(RUNG.rung == rung) & RUNG.source.str.contains("shifted") & RUNG.source.isin(
                [f"{a} shifted" if rung == "R5_input" else f"{a} shifted (mean of 5)" for a in ARMS])]
            mx = nul.groupby("marker").value.max().reindex(MARKERS)
            ax.scatter(x, mx, marker="_", s=300, color="black", zorder=3, label="max of shifted-map nulls")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_ylabel(f"Danenberg 2022\n{rung}\n(95% patient-bootstrap CI)", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1), fontsize=8, frameon=False)
    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(MARKERS, rotation=40, ha="right")
    fig.tight_layout()
    fig.savefig(f"{OUT}/fig_rungs.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

json.dump(DEC, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
print(json.dumps(DEC, indent=1, default=str))
tick("done")